# ☀️ NovaSolar AI — LSTM Solar Energy Forecasting

> **Author:** Avijit Saha Apu | [@191Avi](https://github.com/191Avi)  
> **Project:** [NovaSolar AI](https://github.com/191Avi/Novasolar_ai)  
> **License:** MIT

---

## 🎯 Objective

This notebook trains an **LSTM (Long Short-Term Memory)** deep learning model to forecast solar energy output **24 hours ahead** using historical solar irradiance and weather data.

### Problem Statement
Solar energy output is highly variable — clouds, temperature, and seasonal patterns cause unpredictable fluctuations. Accurate forecasting is critical for:
- Grid stability and load balancing
- Energy trading and monetization
- Reducing curtailment and waste

### Approach
We use a multi-layer LSTM network trained on time-series sequences of solar irradiance, temperature, humidity, and historical power output to predict the next 24 hours of energy generation.

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Generation & Loading](#2-data-generation--loading)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Data Preprocessing](#4-data-preprocessing)
5. [LSTM Model Architecture](#5-lstm-model-architecture)
6. [Training](#6-training)
7. [Evaluation & Results](#7-evaluation--results)
8. [24-Hour Forecast Visualization](#8-24-hour-forecast-visualization)
9. [Model Export](#9-model-export)

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install tensorflow pandas numpy matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print('✅ All imports successful!')

## 2. Data Generation & Loading

For this demonstration, we generate synthetic solar irradiance data that mimics real-world patterns:
- **Seasonal variation** (more sun in summer)
- **Daily cycles** (sunrise/sunset)
- **Weather noise** (clouds, overcast days)
- **Temperature correlation**

> 💡 In production, replace this with real data from: NASA POWER API, PVGIS, OpenWeatherMap, or your IoT sensors.

In [ ]:
def generate_solar_data(days=730, freq='1H'):
    """
    Generate synthetic solar energy dataset with realistic patterns.
    
    Args:
        days: Number of days of data to generate
        freq: Time frequency ('1H' for hourly)
    
    Returns:
        pd.DataFrame: Synthetic solar dataset
    """
    # Create datetime index
    start_date = datetime(2023, 1, 1)
    date_range = pd.date_range(start=start_date, periods=days * 24, freq=freq)
    
    hours = np.arange(len(date_range))
    hour_of_day = date_range.hour
    day_of_year = date_range.dayofyear
    
    # Solar irradiance (W/m²) — follows daily and seasonal patterns
    daily_cycle = np.maximum(0, np.sin(np.pi * (hour_of_day - 6) / 12))
    seasonal_factor = 0.7 + 0.3 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
    cloud_noise = np.random.beta(2, 2, len(date_range))
    irradiance = 1000 * daily_cycle * seasonal_factor * cloud_noise
    
    # Temperature (°C) — correlated with irradiance + seasonal
    base_temp = 15 + 10 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
    temp_noise = np.random.normal(0, 2, len(date_range))
    temperature = base_temp + 0.01 * irradiance + temp_noise
    
    # Humidity (%) — inversely correlated with temperature
    humidity = 60 - 0.3 * temperature + np.random.normal(0, 5, len(date_range))
    humidity = np.clip(humidity, 10, 100)
    
    # Wind speed (m/s)
    wind_speed = np.abs(np.random.normal(3, 1.5, len(date_range)))
    
    # Solar power output (kWh) — derived from irradiance with efficiency
    panel_efficiency = 0.20  # 20% efficiency
    panel_area = 50           # m² of panels
    temp_coefficient = -0.004  # power loss per °C above 25°C
    temp_factor = 1 + temp_coefficient * (temperature - 25)
    power_output = (irradiance * panel_efficiency * panel_area * temp_factor) / 1000
    power_output = np.maximum(0, power_output)
    
    df = pd.DataFrame({
        'datetime': date_range,
        'irradiance': irradiance,
        'temperature': temperature,
        'humidity': humidity,
        'wind_speed': wind_speed,
        'power_output_kwh': power_output
    })
    df.set_index('datetime', inplace=True)
    return df

# Generate 2 years of hourly data
df = generate_solar_data(days=730)

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index[0]} → {df.index[-1]}')
print(f'\nFeatures:\n{df.describe().round(3)}')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Plot 1: Power output over time (one week sample)
sample = df['2023-06-01':'2023-06-14']
axes[0].fill_between(sample.index, sample['power_output_kwh'], alpha=0.7, color='#FF8C00')
axes[0].plot(sample.index, sample['power_output_kwh'], color='#FF6B00', linewidth=0.8)
axes[0].set_title('☀️ Solar Power Output — 2 Week Sample (June 2023)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Power Output (kWh)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Plot 2: Average daily profile
hourly_avg = df.groupby(df.index.hour)['power_output_kwh'].mean()
axes[1].bar(hourly_avg.index, hourly_avg.values, color='#FFD700', alpha=0.85, edgecolor='#FF8C00')
axes[1].set_title('📊 Average Hourly Power Output Profile', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Power Output (kWh)')
axes[1].set_xticks(range(24))

# Plot 3: Monthly average
monthly_avg = df.resample('M')['power_output_kwh'].mean()
axes[2].plot(monthly_avg.index, monthly_avg.values, marker='o', linewidth=2,
             color='#2196F3', markerfacecolor='#FF8C00', markersize=8)
axes[2].fill_between(monthly_avg.index, monthly_avg.values, alpha=0.2, color='#2196F3')
axes[2].set_title('📈 Monthly Average Power Output (Seasonal Pattern)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Avg Power Output (kWh)')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.suptitle('NovaSolar AI — Exploratory Data Analysis', y=1.02, fontsize=15, fontweight='bold')
plt.savefig('eda_solar_data.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

## 4. Data Preprocessing

### Key steps:
1. **Normalize** features to [0, 1] range (critical for LSTM convergence)
2. **Create sequences** — sliding window of past 48 hours to predict next 24 hours
3. **Train/Val/Test split** — 70% / 15% / 15%

In [ ]:
# Configuration
LOOKBACK = 48      # Use past 48 hours as input
HORIZON = 24       # Predict next 24 hours
FEATURES = ['irradiance', 'temperature', 'humidity', 'wind_speed', 'power_output_kwh']
TARGET = 'power_output_kwh'

# --- Normalize ---
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[FEATURES])
target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler.fit_transform(df[[TARGET]])

# --- Create sequences ---
def create_sequences(data, lookback, horizon, target_col_idx=4):
    X, y = [], []
    for i in range(lookback, len(data) - horizon):
        X.append(data[i - lookback:i])          # Past 48h of all features
        y.append(data[i:i + horizon, target_col_idx])  # Next 24h of power output
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOKBACK, HORIZON)

# --- Train / Val / Test split ---
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f'Input shape  (X): {X.shape}  → [samples, timesteps, features]')
print(f'Output shape (y): {y.shape} → [samples, forecast_horizon]')
print(f'\nSplit:')
print(f'  Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/n*100:.0f}%)')
print(f'  Val:   {X_val.shape[0]:,} samples ({X_val.shape[0]/n*100:.0f}%)')
print(f'  Test:  {X_test.shape[0]:,} samples ({X_test.shape[0]/n*100:.0f}%)')

## 5. LSTM Model Architecture

### Architecture Design:
```
Input (48 timesteps × 5 features)
    ↓
LSTM(128) + Dropout(0.2) + BatchNorm
    ↓
LSTM(64)  + Dropout(0.2) + BatchNorm
    ↓
LSTM(32)  + Dropout(0.1)
    ↓
Dense(64, relu)
    ↓
Dense(24)  ← Output: 24-hour forecast
```

In [ ]:
def build_lstm_model(lookback, n_features, horizon):
    """
    Build multi-layer LSTM model for solar energy forecasting.
    
    Args:
        lookback: Input sequence length (timesteps)
        n_features: Number of input features
        horizon: Forecast horizon (output steps)
    
    Returns:
        Compiled Keras model
    """
    model = Sequential([
        # Layer 1: LSTM with return sequences for stacking
        LSTM(128, return_sequences=True,
             input_shape=(lookback, n_features),
             kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        Dropout(0.2),
        BatchNormalization(),
        
        # Layer 2: LSTM
        LSTM(64, return_sequences=True,
             kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        Dropout(0.2),
        BatchNormalization(),
        
        # Layer 3: LSTM (final, no return sequences)
        LSTM(32, return_sequences=False),
        Dropout(0.1),
        
        # Dense layers
        Dense(64, activation='relu'),
        Dense(horizon, activation='linear', name='forecast_output')
    ], name='NovaSolar_LSTM')
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='huber',       # Robust to outliers vs MSE
        metrics=['mae']
    )
    return model

model = build_lstm_model(LOOKBACK, len(FEATURES), HORIZON)
model.summary()

## 6. Training

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_novasolar_lstm.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

# Train
print('🚀 Training NovaSolar LSTM model...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
print('✅ Training complete!')

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', color='#2196F3')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#FF8C00')
axes[0].set_title('Model Loss (Huber)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train MAE', color='#4CAF50')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='#F44336')
axes[1].set_title('Mean Absolute Error', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation & Results

In [ ]:
# Predict on test set
y_pred = model.predict(X_test, verbose=0)

# Inverse transform predictions
def inverse_transform_predictions(scaled_preds, scaler, n_features=5, target_idx=4):
    """Inverse transform only the target feature predictions."""
    dummy = np.zeros((len(scaled_preds), n_features))
    result_list = []
    for i in range(scaled_preds.shape[1]):
        dummy[:, target_idx] = scaled_preds[:, i]
        inv = scaler.inverse_transform(dummy)
        result_list.append(inv[:, target_idx])
    return np.column_stack(result_list)

y_pred_inv = inverse_transform_predictions(y_pred, scaler)
y_test_inv = inverse_transform_predictions(y_test, scaler)

# Flatten for overall metrics
y_pred_flat = y_pred_inv.flatten()
y_test_flat = y_test_inv.flatten()

# Metrics
mae = mean_absolute_error(y_test_flat, y_pred_flat)
rmse = np.sqrt(mean_squared_error(y_test_flat, y_pred_flat))
r2 = r2_score(y_test_flat, y_pred_flat)
mape = np.mean(np.abs((y_test_flat - y_pred_flat) / (y_test_flat + 1e-8))) * 100
accuracy = max(0, (1 - mape / 100)) * 100

print('=' * 50)
print('  📊 NovaSolar LSTM — Model Performance')
print('=' * 50)
print(f'  MAE  (Mean Absolute Error) : {mae:.4f} kWh')
print(f'  RMSE (Root Mean Sq. Error) : {rmse:.4f} kWh')
print(f'  R²   (Coefficient)         : {r2:.4f}')
print(f'  MAPE (Mean Abs. % Error)   : {mape:.2f}%')
print(f'  Forecast Accuracy          : {accuracy:.2f}%')
print('=' * 50)

## 8. 24-Hour Forecast Visualization

In [ ]:
# Visualize a 3-day forecast sample
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

sample_indices = [0, len(X_test)//3, 2*len(X_test)//3]
titles = ['Day 1 Forecast Sample', 'Day 2 Forecast Sample', 'Day 3 Forecast Sample']

for ax, idx, title in zip(axes, sample_indices, titles):
    hours = np.arange(HORIZON)
    actual = y_test_inv[idx]
    predicted = y_pred_inv[idx]
    
    ax.fill_between(hours, actual, alpha=0.3, color='#2196F3', label='Actual')
    ax.plot(hours, actual, 'o-', color='#2196F3', linewidth=2, markersize=4, label='Actual Output')
    ax.plot(hours, predicted, 's--', color='#FF8C00', linewidth=2, markersize=4, label='LSTM Forecast')
    
    # Confidence band
    std_dev = np.abs(actual - predicted) * 0.3
    ax.fill_between(hours, predicted - std_dev, predicted + std_dev,
                    alpha=0.2, color='#FF8C00', label='Uncertainty Band')
    
    ax.set_title(f'☀️ {title} — 24-Hour Ahead Forecast', fontweight='bold', fontsize=12)
    ax.set_xlabel('Hour of Forecast')
    ax.set_ylabel('Power Output (kWh)')
    ax.legend(loc='upper right')
    ax.set_xticks(range(0, 24, 2))
    ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)], rotation=30)

plt.suptitle('NovaSolar AI — LSTM 24-Hour Solar Energy Forecast', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('forecast_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Forecast visualization saved!')

## 9. Model Export

In [ ]:
import os, json

# Save model
os.makedirs('../models', exist_ok=True)
model.save('../models/novasolar_lstm_v1.h5')

# Save scaler config
scaler_config = {
    'feature_names': FEATURES,
    'target_feature': TARGET,
    'lookback_hours': LOOKBACK,
    'forecast_horizon_hours': HORIZON,
    'model_version': '1.0.0',
    'architecture': 'LSTM-128-64-32-Dense64-Dense24',
    'metrics': {
        'mae': round(float(mae), 4),
        'rmse': round(float(rmse), 4),
        'r2': round(float(r2), 4),
        'accuracy_pct': round(float(accuracy), 2)
    }
}
with open('../models/model_config.json', 'w') as f:
    json.dump(scaler_config, f, indent=2)

print('✅ Model exported to ../models/')
print(f'   → novasolar_lstm_v1.h5')
print(f'   → model_config.json')
print(f'\n🌞 NovaSolar AI LSTM training complete!')
print(f'   Forecast Accuracy: {accuracy:.2f}%')
print(f'   Ready for deployment 🚀')

---

## 📌 Next Steps

- [ ] Replace synthetic data with real NASA POWER / PVGIS data
- [ ] Add weather forecast features for improved accuracy
- [ ] Experiment with Transformer / Attention mechanisms
- [ ] Integrate with IoT pipeline for real-time inference
- [ ] Deploy as FastAPI microservice
- [ ] Add uncertainty quantification (conformal prediction)

---

## 🤝 Contributing

Found a way to improve the model? See [CONTRIBUTING.md](../CONTRIBUTING.md)!

⭐ **Star the repo if this was helpful:** [github.com/191Avi/Novasolar_ai](https://github.com/191Avi/Novasolar_ai)

---
*Built with ❤️ by [Avijit Saha Apu](https://github.com/191Avi) | NovaSolar AI © 2026*